<a href="https://colab.research.google.com/github/arthursrosa-star/mapreduce_taxi_nyc/blob/main/MapReduce_PDM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Processamento de Dados Massivos CIADM3B


### Análise de Corridas de Táxi NYC 2024 com MapReduce

Este notebook aplica o paradigma **MapReduce** para responder a um conjunto de perguntas sobre o dataset de corridas de táxi amarelo de Nova York (amostra de 2024).


1.   Número de viagens por tipo de pagamento. (obs. verificar no dicionário de dados o nome do tipo de pagamento)
2. Receita total por tipo de pagamento.  (obs. verificar no dicionário de dados o nome do tipo de pagamento)
3. Tarifa média cobrada nas viagens.
4. Data e hora em que foi feita a viagem mais longa.
5. Quantidade de viagens por hora.
6. Distância total percorrida por hora.




Dataset: `nyc_tripdata_2024_sample_1M.csv`
Dicionário de dados: `data_dictionary_trip_records_yellow.pdf`


## 1. Framework MapReduce

In [1]:
import pandas as pd
from multiprocessing import Pool
from collections import defaultdict

CSV_PATH = "/content/nyc_tripdata_2024_sample_1M.csv"
CHUNKSIZE = 100_000
N_WORKERS = 4

In [2]:
def read_chunks(path, chunksize):
    reader = pd.read_csv(
        path,
        chunksize=chunksize,
        engine="python",
        on_bad_lines="skip",
    )
    for chunk in reader:
        yield chunk


def map_reduce(chunks, map_fn, reduce_fn, n_workers=N_WORKERS):
    with Pool(n_workers) as pool:
        mapped_results = pool.map(map_fn, chunks)

    shuffled = defaultdict(list)
    for pairs in mapped_results:
        for key, value in pairs:
            shuffled[key].append(value)

    return {key: reduce_fn(values) for key, values in shuffled.items()}

## 2. Mapeamento dos tipos de pagamento

Conforme o dicionário de dados, `payment_type` segue a codificação:

| Código | Tipo de pagamento |
|---|---|
| 0 | Flex Fare trip |
| 1 | Credit card |
| 2 | Cash |
| 3 | No charge |
| 4 | Dispute |
| 5 | Unknown |
| 6 | Voided trip |


In [11]:
PAYMENT_TYPES = {
    0: "Flex Fare trip",
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip",
}

## 3. Número de viagens por tipo de pagamento

In [12]:
def map_trip_count(chunk):
    return [(PAYMENT_TYPES.get(pt, "Desconhecido"), 1) for pt in chunk["payment_type"]]


def reduce_sum(values):
    return sum(values)


chunks = list(read_chunks(CSV_PATH, CHUNKSIZE))
trips_by_payment = map_reduce(chunks, map_trip_count, reduce_sum)
pd.Series(trips_by_payment).sort_values(ascending=False)

,0
Credit card,743405
Cash,136221
Flex Fare trip,97124
Dispute,16543
No charge,6707


## 4. Receita total por tipo de pagamento

In [23]:
def map_revenue(chunk):
    return [
        (PAYMENT_TYPES.get(pt, "Desconhecido"), amount)
        for pt, amount in zip(chunk["payment_type"], chunk["total_amount"])
    ]


revenue_by_payment = map_reduce(chunks, map_revenue, reduce_sum)
pd.Series(revenue_by_payment).sort_values(ascending=False)

,0
Credit card,21785219.95
Cash,3168095.90
Flex Fare trip,2376069.77
No charge,53932.48
Dispute,25214.51


## 5. Tarifa média cobrada nas viagens

In [22]:
def map_Tarifa_media(chunk):
    return [("Valor médio da corrida:", (fare, 1)) for fare in chunk["fare_amount"]]


def reduce_average(values):
    total = sum(v[0] for v in values)
    count = sum(v[1] for v in values)
    return total / count


average_fare = map_reduce(chunks, map_Tarifa_media, reduce_average)
average_fare

{'Valor médio da corrida:': 18.86029663}

## 6. Data e hora da viagem mais longa

In [8]:
def map_longest_trip(chunk):
    return [
        ("longest", (dist, pickup))
        for dist, pickup in zip(chunk["trip_distance"], chunk["tpep_pickup_datetime"])
    ]


def reduce_max_distance(values):
    return max(values, key=lambda v: v[0])


longest_trip = map_reduce(chunks, map_longest_trip, reduce_max_distance)
distancia, data_hora = longest_trip["longest"]
print(f"Distância: {distancia} milhas")
print(f"Data e hora do início da viagem: {data_hora}")

Distância: 82831.78 milhas
Data e hora do início da viagem: 2024-03-17 20:02:00


## 7. Quantidade de viagens por hora

In [21]:
def map_trips_per_hour(chunk):
    horas = pd.to_datetime(chunk["tpep_pickup_datetime"]).dt.hour
    return [(hora, 1) for hora in horas]


trips_per_hour = map_reduce(chunks, map_trips_per_hour, reduce_sum)
pd.Series(trips_per_hour).sort_index()

,0
0,29165
1,18822
2,12280
3,8281
4,6054
5,6194
6,13966
7,28065
8,38308
9,42309


## 8. Distância total percorrida por hora

In [10]:
def map_distance_per_hour(chunk):
    horas = pd.to_datetime(chunk["tpep_pickup_datetime"]).dt.hour
    return list(zip(horas, chunk["trip_distance"]))


distance_per_hour = map_reduce(chunks, map_distance_per_hour, reduce_sum)
pd.Series(distance_per_hour).sort_index()

,0
0.0,38784.55
1.0,21281.03
2.0,12531.19
3.0,9741.26
4.0,10364.35
5.0,11716.94
6.0,23289.09
7.0,36092.45
8.0,41678.03
9.0,72042.56
